In [1]:
from pathlib import Path
import pandas as pd

# Find the project from either the repository root or a notebook directory.
project_root = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate / "src" / "data" / "load_data.py").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Open this notebook inside the project repository.")

dataset_path = project_root / "data" / "processed" / "ev_predictive_maintenance_cleaned.csv"
if not dataset_path.is_file():
    raise FileNotFoundError(
        "Create the local processed CSV with python scripts/preprocess_dataset.py from the project root."
    )

df = pd.read_csv(dataset_path)
df.head()

,timestamp,soc,soh,battery_voltage,battery_current,battery_temperature,charge_cycles,motor_temperature,motor_vibration,motor_torque,...,load_weight,driving_speed,distance_traveled,idle_time,route_roughness,rul,failure_probability,maintenance_type,ttf,component_health_score
0,2020-01-01 00:00:00,0.826099,0.941338,210.163881,-22.753095,27.149201,149.190930,48.496049,0.369095,113.435589,...,741.754518,103.421162,66.232383,0.520922,0.225970,260.503381,0,1,111.116697,0.852745
1,2020-01-01 00:15:00,0.064728,0.916059,364.000102,-27.701120,53.655101,171.702388,57.829492,1.449195,105.587160,...,769.134035,46.041935,3.146238,0.844005,0.204350,212.813954,0,2,179.229425,0.827616
2,2020-01-01 00:30:00,0.873643,0.908020,388.855089,-36.646406,29.559090,191.617645,46.518363,1.859045,119.610302,...,917.262931,59.588422,79.909148,0.992405,0.175125,273.394511,0,1,171.852663,0.876887
3,2020-01-01 00:45:00,0.853009,0.916476,370.570602,-37.609429,29.690283,111.881817,54.163681,0.381500,182.535625,...,600.598736,44.222285,0.774000,0.007615,0.213264,229.508442,0,0,165.221328,0.816290
4,2020-01-01 01:00:00,0.947540,0.913206,390.011904,-14.275808,28.864338,163.774377,42.075978,0.433927,173.298044,...,613.153029,41.374684,2.872124,0.771938,0.770257,257.302631,1,0,176.890659,0.744260


In [2]:
print("Rows and columns:", df.shape)
print("Column names:", df.columns.tolist())

Rows and columns: (175393, 30)
Column names: ['timestamp', 'soc', 'soh', 'battery_voltage', 'battery_current', 'battery_temperature', 'charge_cycles', 'motor_temperature', 'motor_vibration', 'motor_torque', 'motor_rpm', 'power_consumption', 'brake_pad_wear', 'brake_pressure', 'reg_brake_efficiency', 'tire_pressure', 'tire_temperature', 'suspension_load', 'ambient_temperature', 'ambient_humidity', 'load_weight', 'driving_speed', 'distance_traveled', 'idle_time', 'route_roughness', 'rul', 'failure_probability', 'maintenance_type', 'ttf', 'component_health_score']


In [3]:
print("COLUMN NAMES:")
for column in df.columns:
    print(column)

print("\nMISSING VALUES:")
print(df.isna().sum())

COLUMN NAMES:
timestamp
soc
soh
battery_voltage
battery_current
battery_temperature
charge_cycles
motor_temperature
motor_vibration
motor_torque
motor_rpm
power_consumption
brake_pad_wear
brake_pressure
reg_brake_efficiency
tire_pressure
tire_temperature
suspension_load
ambient_temperature
ambient_humidity
load_weight
driving_speed
distance_traveled
idle_time
route_roughness
rul
failure_probability
maintenance_type
ttf
component_health_score

MISSING VALUES:
timestamp                 0
soc                       0
soh                       0
battery_voltage           0
battery_current           0
battery_temperature       0
charge_cycles             0
motor_temperature         0
motor_vibration           0
motor_torque              0
motor_rpm                 0
power_consumption         0
brake_pad_wear            0
brake_pressure            0
reg_brake_efficiency      0
tire_pressure             0
tire_temperature          0
suspension_load           0
ambient_temperature       0
ambie

In [4]:
print("Total missing values:", df.isna().sum().sum())
print("Last 8 columns:", ", ".join(df.columns[-8:]))

if "failure_probability" in df.columns:
    print("\nFailure probability summary:")
    print(df["failure_probability"].describe())
    print("Unique values:", df["failure_probability"].nunique())

Total missing values: 0
Last 8 columns: distance_traveled, idle_time, route_roughness, rul, failure_probability, maintenance_type, ttf, component_health_score

Failure probability summary:
count    175393.000000
mean          0.098818
std           0.298418
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max           1.000000
Name: failure_probability, dtype: float64
Unique values: 2


In [5]:
# The label we want to predict
y = df["failure_probability"].astype(int)

# Exclude possible outcome-related columns to reduce leakage.
# Keep timestamp separately for deciding how to split the data.
exclude_columns = [
    "failure_probability",
    "timestamp",
    "rul",
    "maintenance_type",
    "ttf",
    "component_health_score",
]

X = df.drop(columns=exclude_columns)

print("Input shape:", X.shape)
print("\nNon-numeric columns:")
print(X.select_dtypes(exclude="number").columns.tolist())

print("\nTarget counts:")
print(y.value_counts().sort_index())

print("\nFirst timestamps:")
print(df["timestamp"].head())

Input shape: (175393, 24)

Non-numeric columns:
[]

Target counts:
failure_probability
0    158061
1     17332
Name: count, dtype: int64

First timestamps:
0    2020-01-01 00:00:00
1    2020-01-01 00:15:00
2    2020-01-01 00:30:00
3    2020-01-01 00:45:00
4    2020-01-01 01:00:00
Name: timestamp, dtype: str


In [6]:
timestamps = pd.to_datetime(df["timestamp"], errors="raise")
assert timestamps.notna().all(), "Some timestamps are missing."

# Choose a cutoff date at approximately 80% of the data
cutoff = timestamps.sort_values().iloc[int(len(df) * 0.8)]

train_mask = timestamps < cutoff
test_mask = timestamps >= cutoff

X_train = X.loc[train_mask].copy()
X_test = X.loc[test_mask].copy()
y_train = y.loc[train_mask].copy()
y_test = y.loc[test_mask].copy()

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)
print("Cutoff date:", cutoff)

print("\nTraining label counts:")
print(y_train.value_counts().sort_index())

print("\nTesting label counts:")
print(y_test.value_counts().sort_index())

Training shape: (140314, 24)
Testing shape: (35079, 24)
Cutoff date: 2024-01-01 14:30:00

Training label counts:
failure_probability
0    126464
1     13850
Name: count, dtype: int64

Testing label counts:
failure_probability
0    31597
1     3482
Name: count, dtype: int64


In [7]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

model = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    )
)

model.fit(X_train, y_train)

print("Baseline model trained successfully!")

Baseline model trained successfully!


In [8]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("CLASSIFICATION REPORT:")
print(classification_report(y_test, y_pred, digits=3, zero_division=0))

print("CONFUSION MATRIX (rows = actual, columns = predicted):")
print(confusion_matrix(y_test, y_pred, labels=[0, 1]))

print("\nROC-AUC:", round(roc_auc_score(y_test, y_prob), 3))
print("Average precision:", round(average_precision_score(y_test, y_prob), 3))
print("Positive-label prevalence:", round(y_test.mean(), 3))

CLASSIFICATION REPORT:
              precision    recall  f1-score   support

           0      0.899     0.502     0.644     31597
           1      0.098     0.489     0.163      3482

    accuracy                          0.500     35079
   macro avg      0.498     0.495     0.403     35079
weighted avg      0.820     0.500     0.596     35079

CONFUSION MATRIX (rows = actual, columns = predicted):
[[15854 15743]
 [ 1779  1703]]

ROC-AUC: 0.491
Average precision: 0.097
Positive-label prevalence: 0.099


In [9]:
train_prob = model.predict_proba(X_train)[:, 1]

print("Training ROC-AUC:",
      round(roc_auc_score(y_train, train_prob), 3))

print("Training average precision:",
      round(average_precision_score(y_train, train_prob), 3))

print("Training positive rate:",
      round(y_train.mean(), 3))

Training ROC-AUC: 0.512
Training average precision: 0.102
Training positive rate: 0.099


In [10]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

# Sort training rows by time
ordered_index = timestamps.loc[X_train.index].sort_values().index
X_ordered = X_train.loc[ordered_index]
y_ordered = y_train.loc[ordered_index]

# Earlier 80% for learning, later 20% for validation
split = int(len(X_ordered) * 0.8)

X_fit = X_ordered.iloc[:split]
X_val = X_ordered.iloc[split:]
y_fit = y_ordered.iloc[:split]
y_val = y_ordered.iloc[split:]

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    min_samples_leaf=20,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_fit, y_fit)

val_prob = rf_model.predict_proba(X_val)[:, 1]

print("Validation ROC-AUC:",
      round(roc_auc_score(y_val, val_prob), 3))
print("Validation average precision:",
      round(average_precision_score(y_val, val_prob), 3))
print("Validation positive rate:",
      round(y_val.mean(), 3))

Validation ROC-AUC: 0.502
Validation average precision: 0.099
Validation positive rate: 0.098
